In [1]:
import xarray as xr

ds = xr.open_mfdataset("inputs/WHACS/raw/*.nc", engine="h5netcdf")

# Rechunking logic:
# Since you want "one seapoint for all time,"
# you want to increase the time chunk size before writing to Zarr.
ds = ds.chunk({"time": -1, "seapoint": 16})

# Write to Zarr using the cluster
ds.to_zarr("inputs/WHACS/processed/Samoa_WHACS_seapoint.zarr", consolidated=True)

/tmp/ipykernel_47889/2801151476.py:3: FutureWarning: In a future version of xarray the default value for compat will change from compat='no_conflicts' to compat='override'. This is likely to lead to different results when combining overlapping variables with the same name. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set compat explicitly.
  ds = xr.open_mfdataset("inputs/WHACS/raw/*.nc", engine="h5netcdf")
/tmp/ipykernel_47889/2801151476.py:3: FutureWarning: In a future version of xarray the default value for compat will change from compat='no_conflicts' to compat='override'. This is likely to lead to different results when combining overlapping variables with the same name. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set compat explicitly.
  ds = xr.open_mfdataset("inputs/WHACS/raw/*.nc", engine="h5netcdf")
/tmp/ipykernel_47889/2801151476.py

# Download and process WHACS offshore waves

Download WHACS bulk and partition spectra, remap them onto metamodel goals, and
write the processed zarr used by dynamic forcing and hindcast reconstruction.

**Prerequisites:** ``04_Select_Goals_by_Hindcasts.ipynb`` (``Samoa_goals.json``).

## Configuration


In [1]:
import numpy as np
import xarray as xr

whacs_data = xr.open_dataset("inputs/WHACS/processed/Samoa_WHACS_seapoint.zarr").load()
whacs_data = whacs_data.assign({"tp": 1.0 / whacs_data["fp"]})
whacs_data["tp"].attrs = {
    "long_name": "Wave peak period",
    "units": "s",
}
whacs_data["hs"] = np.sqrt(
    whacs_data["phs0"].fillna(0) ** 2
    + whacs_data["phs1"].fillna(0) ** 2
    + whacs_data["phs2"].fillna(0) ** 2
    + whacs_data["phs3"].fillna(0) ** 2
)
whacs_data

<xarray.Dataset> Size: 2GB
Dimensions:                      (time: 394464, seapoint: 16)
Coordinates:
  * time                         (time) datetime64[ns] 3MB 1979-01-01 ... 202...
  * seapoint                     (seapoint) int32 64B 970691 971044 ... 970352
Data variables: (12/42)
    cge                          (time, seapoint) float64 50MB 23.4 ... 10.0
    dir                          (time, seapoint) float64 50MB 348.1 ... 328.9
    dp                           (time, seapoint) float64 50MB 348.0 ... 298.0
    fp                           (time, seapoint) float64 50MB 0.061 ... 0.073
    latitude                     (time, seapoint) float32 25MB -13.22 ... -13.84
    longitude                    (time, seapoint) float32 25MB -177.0 ... -173.1
    ...                           ...
    t01                          (time, seapoint) float64 50MB 8.89 ... 8.91
    t02                          (time, seapoint) float64 50MB 7.68 7.67 ... 7.6
    uwnd                         (time, seapoint) float64 50MB -2.5 ... -0.6
    vwnd                         (time, seapoint) float64 50MB -4.3 ... -1.2
    tp                           (time, seapoint) float64 50MB 16.39 ... 13.7
    hs                           (time, seapoint) float64 50MB 1.789 ... 1.216
Attributes: (12/59)
    Conventions:                     CF-1.11 ACDD-1.3
    title:                           Wave Hindcast for the Australian Climate...
    summary:                         Wave Hindcast for the Australian Climate...
    keywords:                        WHACS, ACS, waves, ocean, coast, WWIII, ...
    id:                              BoM-CSIRO-WHACS-001
    naming_authority:                au.csiro
    ...                              ...
    geospatial_lon_max:              179.9375
    geospatial_vertical_positive:    up
    time_coverage_start:             1979-01-01T00:00:00
    time_coverage_end:               1979-01-31T23:00:00
    time_coverage_duration:          P30DT23H0M0S
    time_coverage_resolution:        P0DT1H0M0S

In [2]:
ordered_seapoints_for_goals = [
    970691,
    971044,
    971049,
    970815,
    970701,
    970364,
    970367,
    970142,
    969669,
    969426,
    969542,
    969780,
    969776,
    969892,
    970007,
    970352,
]

whacs_data = whacs_data.sel(seapoint=ordered_seapoints_for_goals)[
    [
        "hs",
        "tp",
        "dir",
        "spr",
        "phs0",
        "ptp0",
        "pdir0",
        "pspr0",
        "phs1",
        "ptp1",
        "pdir1",
        "pspr1",
        "phs2",
        "ptp2",
        "pdir2",
        "pspr2",
        "phs3",
        "ptp3",
        "pdir3",
        "pspr3",
    ]
].rename({"seapoint": "goal"})
whacs_data["goal"] = range(1, 17)
whacs_data = whacs_data.assign_coords(
    {"seapoint": ("goal", ordered_seapoints_for_goals)}
)
whacs_data.to_zarr("inputs/WHACS/processed/Samoa_WHACS_goal.zarr")

/home/tausiaj/miniforge3/envs/bluemath-311-dev/lib/python3.11/site-packages/zarr/api/asynchronous.py:247: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(


In [3]:
xr.open_zarr("inputs/WHACS/processed/Samoa_WHACS_goal.zarr")

<xarray.Dataset> Size: 1GB
Dimensions:   (time: 394464, goal: 16)
Coordinates:
  * time      (time) datetime64[ns] 3MB 1979-01-01 ... 2023-12-31T23:00:00
  * goal      (goal) int64 128B 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16
    seapoint  (goal) int64 128B dask.array<chunksize=(16,), meta=np.ndarray>
Data variables: (12/20)
    dir       (time, goal) float64 50MB dask.array<chunksize=(394464, 16), meta=np.ndarray>
    hs        (time, goal) float64 50MB dask.array<chunksize=(49308, 2), meta=np.ndarray>
    pdir0     (time, goal) float64 50MB dask.array<chunksize=(394464, 16), meta=np.ndarray>
    pdir1     (time, goal) float64 50MB dask.array<chunksize=(394464, 16), meta=np.ndarray>
    pdir2     (time, goal) float64 50MB dask.array<chunksize=(394464, 16), meta=np.ndarray>
    pdir3     (time, goal) float64 50MB dask.array<chunksize=(394464, 16), meta=np.ndarray>
    ...        ...
    ptp0      (time, goal) float64 50MB dask.array<chunksize=(394464, 16), meta=np.ndarray>
    ptp1      (time, goal) float64 50MB dask.array<chunksize=(394464, 16), meta=np.ndarray>
    ptp2      (time, goal) float64 50MB dask.array<chunksize=(394464, 16), meta=np.ndarray>
    ptp3      (time, goal) float64 50MB dask.array<chunksize=(394464, 16), meta=np.ndarray>
    spr       (time, goal) float64 50MB dask.array<chunksize=(394464, 16), meta=np.ndarray>
    tp        (time, goal) float64 50MB dask.array<chunksize=(49308, 2), meta=np.ndarray>
Attributes: (12/59)
    Conventions:                     CF-1.11 ACDD-1.3
    title:                           Wave Hindcast for the Australian Climate...
    summary:                         Wave Hindcast for the Australian Climate...
    keywords:                        WHACS, ACS, waves, ocean, coast, WWIII, ...
    id:                              BoM-CSIRO-WHACS-001
    naming_authority:                au.csiro
    ...                              ...
    geospatial_lon_max:              179.9375
    geospatial_vertical_positive:    up
    time_coverage_start:             1979-01-01T00:00:00
    time_coverage_end:               1979-01-31T23:00:00
    time_coverage_duration:          P30DT23H0M0S
    time_coverage_resolution:        P0DT1H0M0S